In [2]:
%reset -f

## Imports

In [3]:
%%capture
%pip install --upgrade google-genai

In [4]:
%%capture
%pip install google-cloud-secret-manager

In [5]:
%%capture
%pip install --upgrade google-auth google-cloud-bigquery google-cloud-core

In [6]:
%%capture
import pandas as pd

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import numpy as np 
from google.cloud import bigquery

import time

from google.cloud.exceptions import NotFound
import json
import os
import re
import io
import math

from datetime import timedelta
from dateutil.relativedelta import relativedelta
from datetime import date, timedelta, datetime

from functools import partial
from pathlib import Path

In [7]:
%%capture
# Google packages
from google.cloud import secretmanager, storage
from google.oauth2 import service_account

# Gemini packages
from google import genai
from google.genai import types
import base64

In [8]:
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 1.26.4
Pandas version: 2.2.2


## Configs

In [9]:
# Adjust to your own Google Cloud project
GOOGLE_CLOUD_PROJECT = "pcln-pl-busdatasci-prod"

# gemini model
#model = "gemini-3.5-flash"
model = "gemini-3.1-pro-preview"
#model = "gemini-2.5-flash"

BATCH_SIZE = 25
OUTPUT_CSV = "hotel_amenity_output.csv"
CHECKPOINT_JSON = "hotel_amenity_checkpoint.json"

# Model

## Queries

In [10]:
sql_query = '''

SELECT 
  cs.*,
  hpd.METRO_REGION_NAME as hotel_metro_name,
  amx.amenity_code_list as hotel_amenity_code_list
FROM `pcln-pl-busdatasci-prod.commercial_strategy.commstrat_pcln_hotel_dim_order` cs
INNER JOIN `pcln-pl-data-prod.xprod_mart.hotel_property_dim_v` hpd ON CAST(hpd.hotel_id AS STRING) = cs.hotel_id
LEFT JOIN `pcln-pl-data-prod.hotel_base.hotel_amenity_xref` amx ON CAST(amx.hotel_id AS STRING) = cs.hotel_id AND amx.amenity_source_code = hpd.amenity_source_code
WHERE 1=1
AND cs.hotel_id IS NOT NULL
AND hpd.METRO_REGION_NAME in ("NEW YORK, NY","LAS VEGAS, NV")
ORDER BY gross_units DESC;

'''

In [11]:
bq_client = bigquery.Client(project=GOOGLE_CLOUD_PROJECT)
df = bq_client.query(sql_query).to_dataframe()

In [12]:
# Remove commas completely
df['hotel_name'] = df['hotel_name'].str.replace(',', '', regex=False)

#take out problematic hotel
df = df[df['hotel_id'] != '600235572']

In [13]:
path = Path("hotel_amenity_output.csv")

if path.exists():
    df_csv = pd.read_csv(path)

    if "hotel_id" not in df_csv.columns:
        raise ValueError(f"{path} exists but does not contain a 'hotel_id' column")

    csv_ids = pd.to_numeric(df_csv["hotel_id"], errors="coerce").dropna()
else:
    df_csv = pd.DataFrame()
    csv_ids = pd.Series([], dtype="float64")

# make sure both keys are comparable
df_ids = pd.to_numeric(df["hotel_id"], errors="coerce")

df_remaining = df[~df_ids.isin(csv_ids)].copy()

### Gemini Segmentation Calls

In [10]:
INPUT_COLS = [
    "hotel_id",
    "hotel_name",
    "hotel_latitude",
    "hotel_longitude",
    "brand_owner",
    "hotel_state",
    "hotel_country",
    "chain_scale_segment",
    "hotel_property_type",
    "hotel_address",
    "hotel_zip_code",
    "hotel_metro_name",
    "hotel_amenity_code_list"
]

# -----------------------------
# 1) LLM CALL: refactor generate() to take pipe_table_str
# -----------------------------
def generate(pipe_table_str: str) -> str:

    client = genai.Client(
        vertexai=True,
        project="pcln-pl-busdatasci-prod",
        location="global",
    )

    text1 = types.Part.from_text(text=f"""
        <role>
        You are an expert Hospitality Data Analyst. Your task is to verify, audit, and populate a standardized amenity list for hotels across North America based on provided property details, brand standards, and geographic context. 
        </role>
        
        <instructions>
        1. Analyze the incoming hotel data, specifically leveraging `hotel_name`, `brand_owner`, `chain_scale_segment`, and location data (`hotel_address`, `hotel_metro_name`) to verify amenities.
        2. Information Sourcing Priority: When researching or verifying a property, **always prioritize the hotel's official brand webpage or corporate brand standards first.** Only rely on Online Travel Agencies (OTAs) like Expedia/Booking.com or aggregators like TripAdvisor as a secondary fallback if official brand data is completely unavailable.
        3. The `amenity_code_list` column contains legacy data. Treat it as a helpful baseline but NOT the absolute truth. You must reconcile it:
           - If an amenity is missing from `amenity_code_list` but is a verified brand standard on the official website, flag it as 1.
           - If an amenity is present in `amenity_code_list` but contradicts official property documentation or is geographically impossible (e.g., BCHFRNT in Las Vegas), flag it as 0.
        4. Strict Binary Output: For every amenity column, you must output exactly "1" (True/Present) or "0" (False/Absent). Do not leave blank cells, do not use "N/A", and do not use "Unknown".
        5. Do not hallucinate. If a property's existence or basic brand identity cannot be verified, fallback to standard baseline expectations for that `chain_scale_segment`.
        </instructions>
        
       <amenity_definitions>
        Use these specific criteria to determine binary flags:
        - ADLTONLY: The property is strictly adults-only, or features a prominently marketed adults-only pool/section.
        - AIRSHUTTL: Property provides airport transit (paid or free).
        - BARLOUNG: Has a dedicated bar or lounge area on-site.
        - BCHFRNT: Directly borders a coastal beach or oceanfront. (Always 0 for Las Vegas and NYC).
        - BUSCNTR: Has a dedicated business center or guest computing/printing station.
        - CASINO: Has an active, on-site gambling floor (common in Las Vegas, rare in NYC).
        - CLUB: Features an active, on-site dayclub or nightclub venue attached to the hotel.
        - DIGIKEY: Brand supports mobile app check-in and digital room keys.
        - ECOCERT: Property or brand is explicitly eco-certified/green-certified.
        - EVCHARGE: On-site electric vehicle charging stations are available.
        - FAIRSHUTTL: Property provides complimentary airport transit. (If FAIRSHUTTL is 1, AIRSHUTTL must be 1).
        - FBRKFST: Offers complimentary breakfast to all guests.
        - FITSPA: Property features a gym/fitness center AND/OR spa facilities.
        - FPRKING: Self-parking is 100% complimentary. (Rarely 1 in NYC).
        - HANDFAC: Features ADA-compliant accessible facilities/rooms.
        - HOTTUB: Features a hot tub, jacuzzi, or whirlpool available for guest use.
        - KITCHEN: Rooms/Suites feature dedicated kitchenettes or full kitchens.
        - LAUNDRY: Guest laundry facilities or valet dry cleaning service is available.
        - NSMKFAC: The entire property enforces a 100% non-smoking policy.
        - PETALLOW: The property allows pets (regardless of deposit/fee).
        - RESTRNT: Has at least one sit-down restaurant on property.
        - ROOMSRVC: Offers in-room food delivery service.
        - SPA: Offers formal spa services, massages, or wellness treatments on property.
        - SPOOL: Has an operational swimming pool (indoor or outdoor).
        - VALETPRK: Offers valet parking services.
        - GOLF: Property features an on-site 18-hole golf course OR offers exclusive tee-time access to an affiliated private course.
        - WATRFRNT: Property is located directly on a waterfront (lake, river, bay, harbor). (Note: A property can be WATRFRNT without being BCHFRNT).
        </amenity_definitions>
        
        <data>
        The data string is pipe-delimited with the following schema:
        hotel_id|hotel_name|hotel_latitude|hotel_longitude|brand_owner|hotel_state|hotel_country|chain_scale_segment|hotel_property_type|hotel_address|hotel_zip_code|hotel_metro_name|amenity_code_list
        
        {pipe_table_str}
        </data>
        
        <output_format>
        Provide a Markdown Table with exactly the columns listed below. 
        Do not output introductory text, greetings, or explanations before the markdown table. 
        Print exactly 'Output Complete' on a new line immediately after the table ends.
        
        | hotel_id | hotel_name | hotel_metro_name | ADLTONLY | AIRSHUTTL | BARLOUNG | BCHFRNT | BUSCNTR | CASINO | CLUB | DIGIKEY | ECOCERT | EVCHARGE | FAIRSHUTTL | FBRKFST | FITSPA | FPRKING | GOLF | HANDFAC | HOTTUB | KITCHEN | LAUNDRY | NSMKFAC | PETALLOW | RESTRNT | ROOMSRVC | SPA | SPOOL | VALETPRK | WATRFRNT |
        </output_format>
        """)

    si_text1 = "You are an expert Hospitality Data Analyst."

    contents = [types.Content(role="user", parts=[text1])]

    generate_content_config = types.GenerateContentConfig(
        temperature=0,
        top_p=1,
        max_output_tokens=60192,
        safety_settings=[
            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
        ],
        system_instruction=[types.Part.from_text(text=si_text1)],
    )

    full_output = ""
    for chunk in client.models.generate_content_stream(
        model=model,
        contents=contents,
        config=generate_content_config,
    ):
        if chunk.text:
            full_output += chunk.text

    return full_output

# -----------------------------
# 2) PARSE markdown -> DataFrame (your logic, wrapped)
# -----------------------------
def parse_llm_markdown(email_summary: str) -> pd.DataFrame:
    # Validate marker
    if not re.search(r"(?mi)^\s*Output Complete\s*$", email_summary.strip()):
        raise ValueError("Missing required 'Output Complete' marker. Failing as requested.")

    # Remove marker
    table_text = re.sub(r"(?mi)^\s*Output Complete\s*$", "", email_summary).strip()

    # Parse markdown table
    lines = [ln for ln in table_text.splitlines() if ln.strip()]
    lines = [ln for ln in lines if not re.match(r"^\s*\|\s*:?-{3,}:?\s*\|", ln)]  # drop alignment row
    cleaned = "\n".join(lines)

    df_out = pd.read_csv(
        io.StringIO(cleaned),
        sep=r"\s*\|\s*",
        engine="python"
    )

    # Drop empty cols created by leading/trailing pipes
    df_out = df_out.loc[:, ~df_out.columns.str.match(r"^Unnamed")]

    # Tidy
    df_out.columns = [c.strip() for c in df_out.columns]
    if "hotel_id" in df_out.columns:
        df_out["hotel_id"] = pd.to_numeric(df_out["hotel_id"], errors="coerce").astype("Int64")

    return df_out

# -----------------------------
# 3) Build pipe-table string for a batch
# -----------------------------
def make_pipe_table_str(batch_df: pd.DataFrame) -> str:
    batch_df = batch_df.copy()
    batch_df = batch_df[INPUT_COLS]  # enforce order

    header = "|".join(batch_df.columns)
    rows = batch_df.fillna("").astype(str).agg("|".join, axis=1)
    return header + "\n" + "\n".join(rows.tolist())

# -----------------------------
# 4) Resume logic: skip already processed hotel_ids
# -----------------------------
def get_already_processed_ids(output_csv: str) -> set:
    if not os.path.exists(output_csv):
        return set()
    try:
        existing = pd.read_csv(output_csv)
        # Check
        if "hotel_id" in existing.columns:
            return set(pd.to_numeric(existing["hotel_id"], errors="coerce").dropna().astype(int).tolist())
        return set()
    except Exception:
        # If file is corrupted/partial, you can manually fix or delete it.
        return set()

# -----------------------------
# 5) Main loop
# -----------------------------
def run_batches(df: pd.DataFrame):
    # Ensure required cols exist
    missing = [c for c in INPUT_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Input df missing columns: {missing}")

    # Normalize hotel_id for matching
    df2 = df.copy()
    df2["hotel_id"] = pd.to_numeric(df2["hotel_id"], errors="coerce").astype("Int64")
    df2 = df2.dropna(subset=["hotel_id"]).copy()
    df2["hotel_id_int"] = df2["hotel_id"].astype(int)

    processed_ids = get_already_processed_ids(OUTPUT_CSV)
    to_do = df2[~df2["hotel_id_int"].isin(processed_ids)].copy()

    total_rows = len(df2)
    remaining_rows = len(to_do)

    if remaining_rows == 0:
        print(f"Nothing to do. {len(processed_ids)} hotels already in {OUTPUT_CSV}.")
        return

    total_batches = math.ceil(remaining_rows / BATCH_SIZE)
    print(f"Total hotels: {total_rows} | Remaining: {remaining_rows} | Batches to run: {total_batches}")

    # Iterate in batches
    for batch_idx, start in enumerate(range(0, remaining_rows, BATCH_SIZE), start=1):
        batch = to_do.iloc[start:start + BATCH_SIZE].copy()

        # Progress tracker
        done_batches = batch_idx - 1
        print(f"\nBatch {batch_idx}/{total_batches} | "
              f"Hotels in batch: {len(batch)} | "
              f"Completed batches: {done_batches} | Remaining batches: {total_batches - done_batches}")

        # 1) pipe string
        pipe_table_str = make_pipe_table_str(batch)

        # 2) LLM call
        try:
            llm_raw = generate(pipe_table_str)
        except Exception as e:
            print(f"❌ LLM call failed on batch {batch_idx}: {e}")
            # Save checkpoint and stop (so you can rerun)
            with open(CHECKPOINT_JSON, "w") as f:
                json.dump({"last_completed_batch": batch_idx - 1}, f, indent=2)
            raise

        # 3) parse back to df
        try:
            df_out = parse_llm_markdown(llm_raw)
        except Exception as e:
            print(f"❌ Parsing failed on batch {batch_idx}: {e}")
            # Optionally dump the raw output for debugging
            debug_path = f"llm_raw_batch_{batch_idx}.txt"
            with open(debug_path, "w") as f:
                f.write(llm_raw)
            print(f"Saved raw LLM output to {debug_path}")
            with open(CHECKPOINT_JSON, "w") as f:
                json.dump({"last_completed_batch": batch_idx - 1}, f, indent=2)
            raise

        # Optional: add datasource/metadata, or join back to input fields here if desired

        # Append to CSV (safe progress after each batch)
        write_header = not os.path.exists(OUTPUT_CSV)
        df_out.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)

        # Update checkpoint
        with open(CHECKPOINT_JSON, "w") as f:
            json.dump(
                {
                    "last_completed_batch": batch_idx,
                    "batches_total": total_batches,
                    "rows_total": int(total_rows),
                    "rows_remaining_at_start": int(remaining_rows),
                    "output_csv": OUTPUT_CSV,
                },
                f,
                indent=2
            )

        print(f"✅ Saved batch {batch_idx} to {OUTPUT_CSV} (rows written: {len(df_out)})")

        # Optional: light throttling
        time.sleep(0.5)

    print(f"\nAll done. Results appended to: {OUTPUT_CSV}")



In [24]:
# ---- Run it ----
run_batches(df_remaining)

Total hotels: 783 | Remaining: 783 | Batches to run: 32

Batch 1/32 | Hotels in batch: 25 | Completed batches: 0 | Remaining batches: 32
✅ Saved batch 1 to hotel_amenity_output.csv (rows written: 25)

Batch 2/32 | Hotels in batch: 25 | Completed batches: 1 | Remaining batches: 31
✅ Saved batch 2 to hotel_amenity_output.csv (rows written: 25)

Batch 3/32 | Hotels in batch: 25 | Completed batches: 2 | Remaining batches: 30
✅ Saved batch 3 to hotel_amenity_output.csv (rows written: 25)

Batch 4/32 | Hotels in batch: 25 | Completed batches: 3 | Remaining batches: 29
✅ Saved batch 4 to hotel_amenity_output.csv (rows written: 25)

Batch 5/32 | Hotels in batch: 25 | Completed batches: 4 | Remaining batches: 28
✅ Saved batch 5 to hotel_amenity_output.csv (rows written: 25)

Batch 6/32 | Hotels in batch: 25 | Completed batches: 5 | Remaining batches: 27
✅ Saved batch 6 to hotel_amenity_output.csv (rows written: 25)

Batch 7/32 | Hotels in batch: 25 | Completed batches: 6 | Remaining batches: 26

### BQ output

In [25]:
# Set up your Google Cloud project ID
project_id = 'pcln-pl-busdatasci-prod'
dataset_id = 'commercial_strategy'
table_id = 'hotel_amenity_boolean_lv_ny_v3'
table_full_path = f'{project_id}.{dataset_id}.{table_id}'

# Initialize a client
client = bigquery.Client(project=project_id)

# Ensure the dataset exists (optional, create if not existing)
dataset_ref = bigquery.DatasetReference(project_id, dataset_id)
dataset = bigquery.Dataset(dataset_ref)
try:
    client.get_dataset(dataset_ref)  # Make an API request.
except NotFound:
    print('Check dataset_id')

In [26]:
path = "hotel_amenity_output.csv"
df_csv = pd.read_csv(path)


In [27]:
import pandas_gbq
pandas_gbq.to_gbq(df_csv, table_full_path, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 11366.68it/s]
